In [1]:
from Bio import SeqIO
import gzip
import re
import pandas as pd


In [2]:


# -------- Helper Functions --------

def motif_match_with_N(motif, seq_window, max_mismatches=0):
    mismatches = 0
    for m, s in zip(motif, seq_window):
        if m == "N":
            continue  # N matches anything
        if m != s:
            mismatches += 1
            if mismatches > max_mismatches:
                return False, mismatches
    return True, mismatches


def reverse_complement(seq):
    complement = str.maketrans("ATCGN", "TAGCN")
    return seq.translate(complement)[::-1]

# -------- Load Genome --------
def load_genome(fasta_path):
    genome = {}
    with gzip.open(fasta_path, "rt") if fasta_path.endswith(".gz") else open(fasta_path) as handle:
        for record in SeqIO.parse(handle, "fasta"):
            genome[record.id] = str(record.seq).upper()
    return genome

# -------- Parse GTF (get TSS) --------
def parse_gff_tss(gff_path):
    tss_list = []
    with gzip.open(gff_path, "rt") if gff_path.endswith(".gz") else open(gff_path) as f:
        for line in f:
            if line.startswith("#"): continue
            parts = line.strip().split("\t")
            if len(parts) < 9: continue
            if parts[2] != "gene": continue  # Only keep "gene" features

            chrom = parts[0]
            start = int(parts[3])
            end = int(parts[4])
            strand = parts[6]
            tss = start if strand == "+" else end

            # Parse attributes for GFF3: key=value;key2=value2
            attr_field = parts[8]
            attr_dict = dict(
                item.split("=", 1) for item in attr_field.strip().split(";") if "=" in item
            )

            # Try to get the gene name from 'Name' or 'gene' field
            gene_id = attr_dict.get("Name") or attr_dict.get("gene") or "unknown"

            tss_list.append((chrom, tss, strand, gene_id))
    return tss_list

# -------- Get most upstream TSS --------
def get_first_tss_per_gene(tss_list):
    first_tss = {}
    for chrom, tss, strand, gene_id in tss_list:
        if gene_id not in first_tss:
            first_tss[gene_id] = (chrom, tss, strand)
        else:
            _, prev_tss, prev_strand = first_tss[gene_id]
            # Compare coordinates depending on strand
            if strand == "+" and tss < prev_tss:
                first_tss[gene_id] = (chrom, tss, strand)
            elif strand == "-" and tss > prev_tss:
                first_tss[gene_id] = (chrom, tss, strand)
    return first_tss

    
# -------- Scan for Motif --------
def find_motifs_near_tss(genome, tss_list, motif, max_mismatches, upstream_start, upstream_end):
    motif_hits = []
    motif_len = len(motif)

    for chrom, tss, strand, gene_id in tss_list:
        if chrom not in genome:
            continue
        seq = genome[chrom]
        if strand == "+":
            region_start = max(0, tss + upstream_start)
            region_end = max(0, tss + upstream_end)
            region = seq[region_start:region_end]
        else:
            region_start = max(0, tss - upstream_end)
            region_end = max(0, tss - upstream_start)
            region = reverse_complement(seq[region_start:region_end])
        
        for i in range(0, len(region) - motif_len + 1):
            window = region[i:i+motif_len]
            matched, mismatches = motif_match_with_N(motif, window, max_mismatches)
            if matched:
                hit_position = tss + (i + upstream_start if strand == "+" else -i - upstream_start)
                # Compute absolute motif position first
                motif_start_abs = tss + (i + upstream_start if strand == "+" else -i - upstream_start)

                # Then compute relative position to TSS
                relative_pos = motif_start_abs - tss
                motif_hits.append({
                    "gene_id": gene_id,
                    "chrom": chrom,
                    "tss": tss,
                    "strand": strand,
                    "motif_start": hit_position,
                    "rel_position_motif": relative_pos,
                    "mismatches": mismatches,
                    "matched_seq": window
                })

    return motif_hits

# -------- Extract TSS sequences --------
def extract_tss_sequences(genome, tss_list, upstream_start, upstream_end):
    tss_sequences = {}
    for chrom, tss, strand, gene_id in tss_list:
        if chrom not in genome:
            continue
        seq = genome[chrom]
        # Extract region according to strand
        if strand == "+":
            region_start = max(0,tss + upstream_start)
            region_end = max(0,tss + upstream_end)
            region = seq[region_start:region_end]
        else:
            region_start = max(0, tss - upstream_end)
            region_end = max(0, tss - upstream_start)
            region = reverse_complement(seq[region_start:region_end])

        tss_sequences[gene_id] = {"chrom": chrom,"tss": tss,"strand": strand,"sequence": region}

    return tss_sequences

# -------- Save TSS sequences as FASTA --------
def save_tss_fasta(tss_sequences, output_file):
    with open(output_file, "w") as file:
        for gene_id, data in tss_sequences.items():
            file.write(f">{gene_id}\n")
            file.write(f"{data['sequence']}\n")

# -------- Filter DESeq file --------
def filter_deseq(csv_path, de_threshold):
    deseq_file = pd.read_csv(csv_path)

    down_genes_table = deseq_file[(deseq_file["pvalue"] < 0.05) & (deseq_file["log2FoldChange"] < de_threshold)]
    down_genes_list = down_genes_table["gene_id"].tolist()
    
    up_genes_table = deseq_file[(deseq_file["pvalue"] < 0.05) & (deseq_file["log2FoldChange"] > de_threshold)]
    up_genes_list = up_genes_table["gene_id"].tolist() 

    return down_genes_list, up_genes_list


##### Run motif analysis
def run_motif_analysis(genome, tss_list, motif, max_mismatches, upstream_start, upstream_end, output_file):
    hits = find_motifs_near_tss(genome, tss_list, motif, max_mismatches, upstream_start, upstream_end)
    df = pd.DataFrame(hits)
    df.to_csv(output_file, index=False)
    print(
        f"Saved {len(df)} motif hits "
        f"to {output_file}"
    )
    return df


##### Run TSS extraction for selected genes
def run_tss_extraction(genome, tss_list, gene_list, upstream_start, upstream_end, output_file):
    # Select genes of interest
    selected_tss = [tss for tss in tss_list if tss[3] in gene_list]
    # Keep only the first TSS per gene
    first_tss = get_first_tss_per_gene(selected_tss)
    selected_tss = [(chrom, tss, strand, gene_id) for gene_id, (chrom, tss, strand) in first_tss.items()]
    # Extract sequences
    sequences = extract_tss_sequences(genome, selected_tss, upstream_start, upstream_end)
    # Save FASTA
    save_tss_fasta(sequences,output_file)
    # Report missing genes
    found_genes = set(sequences.keys())
    missing_genes = [gene for gene in gene_list if gene not in found_genes]
    print(f"Extracted: {len(sequences)} genes")
    print(f"Missing: {len(missing_genes)} genes")

    if missing_genes:
        print("Missing genes:")
        print(", ".join(missing_genes))
    return sequences

In [3]:
# -------- Parameters --------

motif = "CGAACNNNNGTTCG"
max_mismatches = 2
upstream_start = -400
upstream_end = 10

# -------- Input files --------
genome_fasta = "AM1001-oriented.fasta"
gff_file = "annot.gff"
deseq_file = ("../results/deseq2/condition_UV_treated_vs_log_control_deseq2_results.csv")

# -------- Gene list --------
# SOS box, 1 mismatch, logFC > 0
gene_list = ["uvrB","pgaptmp_000491","pgaptmp_000620","pgaptmp_000836","pgaptmp_000965","recA",
    "pgaptmp_001744","lexA","pgaptmp_001866","pgaptmp_002002","pgaptmp_002354","pgaptmp_003546"]

# -------- Load genome and annotation  --------
genome = load_genome(genome_fasta)
tss_list = parse_gff_tss(gff_file)

print(f"Genome sequences loaded: {len(genome)}")
print(f"Gene entries found in GFF: {len(tss_list)}")

Genome sequences loaded: 1
Gene entries found in GFF: 3809


In [4]:
# -------- ANALYSIS --------
# 1. Find motifs
motif_df = run_motif_analysis(genome=genome,
    tss_list=tss_list,
    motif=motif,
    max_mismatches=max_mismatches,
    upstream_start=upstream_start,
    upstream_end=upstream_end,
    output_file="motif_hits.csv"
)
motif_df.head()

Saved 433 motif hits to motif_hits.csv


,gene_id,chrom,tss,strand,motif_start,rel_position_motif,mismatches,matched_seq
0,dnaN,tig00000001_polypolish,1526,+,1126,-400,2,CGAAAAAGCGTACG
1,rlbA,tig00000001_polypolish,2797,+,2463,-334,2,CGAAGCTGTGCTCG
2,pgaptmp_000024,tig00000001_polypolish,26638,-,26796,158,2,CGATCGGCACTTCG
3,pgaptmp_000028,tig00000001_polypolish,29402,+,29187,-215,2,CGAATCAATGCTCG
4,darA,tig00000001_polypolish,40695,+,40477,-218,2,CACACGAAAGTTCG


In [7]:
# 2. Extract TSS sequences from gene_list
tss_sequences = run_tss_extraction(
    genome=genome,
    tss_list=tss_list,
    gene_list=gene_list,
    upstream_start=upstream_start,
    upstream_end=upstream_end,
    output_file="gene_list.fasta"
)

Extracted: 12 genes
Missing: 0 genes
